DATASET:House Prices (Kaggle)

### Dataset Overview:House Prices(Kaggle)
The dataset contains housing-related features such as area, quality, and location.
The target variable is SalePrice. Initial exploration helps identify missing values
and understand data distribution.

In [16]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
Question 1:Load the dataset and create a Series of SalePrice with house IDs as index. Analyze the price distribution and identify the most frequent range.
##Code:
price_series = pd.Series(df['SalePrice'].values, index=df['Id'])
price_series.describe()
price_series.value_counts(bins=10)

(106910.0, 178920.0]              723
(178920.0, 250930.0]              373
(34179.899000000005, 106910.0]    148
(250930.0, 322940.0]              135
(322940.0, 394950.0]               51
(394950.0, 466960.0]               19
(466960.0, 538970.0]                4
(538970.0, 610980.0]                3
(610980.0, 682990.0]                2
(682990.0, 755000.0]                2
Name: count, dtype: int64

### Analysis:
The mean SalePrice (180,000) is higher than the median (163,000), indicating a right-skewed distribution.
Most houses fall within the price range of approximately 120,000–200,000, as seen from the highest frequency bin.
This shows that mid-range houses dominate the dataset, while high-end properties are fewer but increase the average price.

In [7]:
Q2. Determine which five features have the strongest correlation with SalePrice using ufuncs and a correlation matrix. Justify your feature selection with statistical reasoning.
#Code
corr = df.corr(numeric_only=True)
top5 = corr['SalePrice'].sort_values(ascending=False)[1:6]
top5

OverallQual    0.790982
GrLivArea      0.708624
GarageCars     0.640409
GarageArea     0.623431
TotalBsmtSF    0.613581
Name: SalePrice, dtype: float64

### Analysis:
The feature OverallQual shows a strong positive correlation (0.79) with SalePrice, indicating that higher quality homes are significantly more expensive.
Similarly, GrLivArea (0.71) also has a strong influence, suggesting that larger living spaces increase property value.
These high correlation values confirm that both size and quality are key drivers of house prices.

In [9]:
Q3. Apply hierarchical indexing on Neighborhood and BldgType. Examine median sale prices at each hierarchical level using xs() and groupby for multi-level insights.
#Code
grouped = df.groupby(['Neighborhood','BldgType'])['SalePrice'].median()
grouped.head()
grouped.xs('1Fam', level='BldgType')

Neighborhood
Blmngtn    159895.0
BrkSide    124300.0
ClearCr    200250.0
CollgCr    200320.5
Crawfor    200624.0
Edwards    118000.0
Gilbert    181000.0
IDOTRR     104750.0
Mitchel    146900.0
NAmes      141000.0
NWAmes     185000.0
NoRidge    301500.0
NridgHt    335000.0
OldTown    117000.0
SWISU      138475.0
Sawyer     135000.0
SawyerW    189000.0
Somerst    244600.0
StoneBr    377426.0
Timber     228000.0
Veenker    190500.0
Name: SalePrice, dtype: float64

### Analysis:
Hierarchical indexing was applied using Neighborhood and BldgType to understand how location and building type jointly influence house prices.
The median SalePrice was calculated to reduce the impact of extreme values and provide a more stable measure of central tendency.
Using xs(), we extracted data for a specific building type (1Fam), allowing us to compare how different neighborhoods perform within the same housing category.
The results show that location significantly affects property prices, even for similar building types. Some neighborhoods consistently have higher median values, indicating stronger demand or better infrastructure.

In [10]:
Q4. Create a Value Index combining LotArea, OverallQual, and GrLivArea using normalized ufunc transformations. Rank houses and construct a premium property identification report.#Code
df['LotArea_n'] = (df['LotArea'] - df['LotArea'].mean())/df['LotArea'].std()
df['OverallQual_n'] = (df['OverallQual'] - df['OverallQual'].mean())/df['OverallQual'].std()
df['GrLivArea_n'] = (df['GrLivArea'] - df['GrLivArea'].mean())/df['GrLivArea'].std()

df['ValueIndex'] = df[['LotArea_n','OverallQual_n','GrLivArea_n']].mean(axis=1)
df.sort_values(by='ValueIndex', ascending=False)[['Id','ValueIndex']].head()

,Id,ValueIndex
313,314,7.384364
1298,1299,5.340126
249,250,5.333498
335,336,5.054402
523,524,3.932765


### Analysis:
The Value Index was constructed by normalizing and combining key features representing land size, living space, and overall quality.
Normalization ensures that no single feature disproportionately influences the index due to scale differences. The resulting composite score provides a balanced measure of property value.
High-ranking properties in this index typically exhibit a combination of large area and superior quality, making them premium assets. This method simulates real-world valuation techniques used in property assessment.

In [13]:
Q5. Identify features with more than 40% missing values. Examine the trade-off between dropping these features and applying creative imputation strategies. Justify your approach with statistical evidence.
#Code
missing = df.isnull().mean()*100
missing[missing > 40]

Alley          93.767123
MasVnrType     59.726027
FireplaceQu    47.260274
PoolQC         99.520548
Fence          80.753425
MiscFeature    96.301370
dtype: float64

### Analysis:
Features with more than 40% missing values pose a significant challenge in data analysis.
Dropping such features simplifies the dataset and reduces noise, but may eliminate potentially meaningful variables. Conversely, imputing missing values preserves the dataset structure but introduces assumptions that may bias results.
A strategic decision must be made based on feature relevance. In practice, features with high missing rates and low predictive importance are better removed, while critical features should be retained and carefully imputed.

In [8]:
Q6. Using boolean indexing, extract houses with OverallQual of 8 or above and a SalePrice below the median. Examine what makes these properties undervalued using index-aligned operations.
#Code
median_price = df['SalePrice'].median()

undervalued = df[(df['OverallQual'] >= 8) & (df['SalePrice'] < median_price)]

undervalued[['Id','OverallQual','SalePrice']].head()

,Id,OverallQual,SalePrice
458,459,8,161000
1298,1299,10,160000
1324,1325,8,147000
1349,1350,8,122000


### Analysis:
The median SalePrice is approximately 163,000. However, the filtered houses have prices below this value despite having high OverallQual (≥ 8).
This indicates that these properties are undervalued.
On comparing features, some houses may have smaller living areas or fewer garages, which could explain their lower price despite good quality.

In [9]:
Q7. Design a property classification function using apply() that labels houses as Luxury,Mid-Range, or Affordable. Build a neighborhood-wise price segment report using pivot_table.
#Code
def classify(p):
    if p > 300000:
        return "Luxury"
    elif p > 150000:
        return "Mid-Range"
    else:
        return "Affordable"

df['Category'] = df['SalePrice'].apply(classify)

pd.pivot_table(df, values='SalePrice',
               index='Neighborhood',
               columns='Category',
               aggfunc='mean')

Category,Affordable,Luxury,Mid-Range
Neighborhood,,,
Blmngtn,NaN,NaN,194870.882353
Blueste,124000.000000,NaN,151000.000000
BrDale,104493.750000,NaN,NaN
BrkSide,110061.702128,NaN,187952.272727
ClearCr,135810.666667,315000.000000,213669.565217
CollgCr,135193.055556,356935.000000,212728.872727
Crawfor,130098.700000,349016.666667,209907.828571
Edwards,112578.395062,320000.000000,187951.111111
Gilbert,143000.000000,348750.000000,190026.746667


### Analysis:
The classification of houses into Luxury, Mid-Range, and Affordable segments simplifies the complexity of continuous price data into meaningful categories.
The pivot table reveals how these segments are distributed across neighborhoods, highlighting geographic pricing disparities.
Some neighborhoods are dominated by high-end properties, while others cater to mid-range or affordable segments. This segmentation provides valuable insights into market structure and can assist in targeted real estate strategies.

In [5]:
Q8. Examine how different null-filling strategies —mode for categorical columns and median for numerical columns — affect the distribution of SalePrice and related features.
#Code
df_filled = df.copy()
# Fill numerical columns with median
df_filled = df_filled.fillna(df_filled.median(numeric_only=True))

# Fill categorical columns with mode (FIXED)
for col in df_filled.select_dtypes(include='object'):
    df_filled[col] = df_filled[col].fillna(df_filled[col].mode()[0])

# Compare distributions
df['SalePrice'].describe()
df_filled['SalePrice'].describe()

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

### Analysis:
Handling missing values using median (for numerical) and mode (for categorical) ensures that data integrity is maintained without introducing extreme distortions.
The comparison of distributions before and after imputation shows minimal variation, indicating that the chosen methods preserve the underlying data structure.
This approach balances accuracy and completeness, making the dataset more suitable for analysis while avoiding the risks associated with excessive data removal.

In [11]:
Q9. Perform operations between two DataFrames:houses built before 1980 and houses built after 2000.Use index alignment to compare average quality
scores and sale prices.
#Code
old = df[df['YearBuilt'] < 1980]
new = df[df['YearBuilt'] > 2000]

old[['OverallQual','SalePrice']].mean()
new[['OverallQual','SalePrice']].mean()

OverallQual         7.436813
SalePrice      244527.458791
dtype: float64

### Analysis:
The comparison between older and newer houses highlights a clear trend: newer properties generally exhibit higher quality and command higher prices.
This reflects advancements in construction technology, modern design preferences, and improved amenities in recent developments.
The analysis demonstrates how the age of a property significantly influences its market value, making it an important factor in real estate decision-making.

In [14]:
Q10. Build a multi-level real estate analytics report covering null handling, hierarchical grouping, ufunc price modeling, and neighborhood comparisons. Push the notebook to GitHub.
#Code
summary = {
    "Top Features": top5.index.tolist(),
    "High Missing Columns": missing[missing > 40].index.tolist(),
    "Median Price": df['SalePrice'].median(),
    
    # Hierarchical grouping (Q3)
    "Top Neighborhood Prices": df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).head(5).to_dict(),
    
    # Value Index (Q4)
    "Top Value Houses": df.sort_values(by='ValueIndex', ascending=False)['Id'].head(5).tolist(),
    
    # Old vs New comparison (Q9)
    "Old vs New Price": {
        "Old": df[df['YearBuilt'] < 1980]['SalePrice'].mean(),
        "New": df[df['YearBuilt'] > 2000]['SalePrice'].mean()
    }
}
summary

{'Top Features': ['OverallQual',
  'GrLivArea',
  'GarageCars',
  'GarageArea',
  'TotalBsmtSF'],
 'High Missing Columns': ['Alley',
  'MasVnrType',
  'FireplaceQu',
  'PoolQC',
  'Fence',
  'MiscFeature'],
 'Median Price': 163000.0,
 'Top Neighborhood Prices': {'NridgHt': 315000.0,
  'NoRidge': 301500.0,
  'StoneBr': 278000.0,
  'Timber': 228475.0,
  'Somerst': 225500.0},
 'Top Value Houses': [314, 1299, 250, 336, 524],
 'Old vs New Price': {'Old': np.float64(142987.92806603774),
  'New': np.float64(244527.4587912088)}}

### Analysis:
This code constructs a multi-level real estate analytics report by combining insights from different parts of the dataset.
It includes feature importance using correlation, identification of columns with high missing values, and central price estimation using median. 
Hierarchical grouping is applied through neighborhood-wise median prices, allowing comparison across locations. The Value Index is used to identify premium properties based on combined features.
Additionally, a comparison between older and newer houses highlights how construction year impacts pricing.
Overall, this integrated approach provides a comprehensive and structured understanding of factors influencing real estate prices.